The full analysis can be found in the following link:   

https://github.com/Eduardo-95-DS/cloud-walk-fraud-case/blob/main/case.ipynb

# Suspicious Behaviors Identified & Proposed Actions

## Dataset Overview

The dataset contains **3.199 card-not-present (CNP) transactions** from November 1st to December 1st, with the Black Friday period accounting for 63.93% of all the transactions.

The fraud rate (chargeback) is **12.22%** (391 transactions), which is high, mainly because of the Black Friday period.

Key entities: 2.704 users, 2.925 cards (410 BINs), 1.756 merchants, and 1.996 devices 
(830 transactions with missing device_id, replaced by artificially-created ids). 

## Business Assumptions

The following assumptions guided the analysis and shaped the detection rules:    

False negatives (missed fraud that results in chargebacks) are costlier than false positives (blocked legitimate transactions).     
Chargebacks mean direct monetary loss and false positives can cause lost sales and customer friction in the long-term, so the solution prioritizes maximizing maximizing recall while keeping precision above an acceptable level.

Period-specific analysis: all findings separate Black Friday (Nov 22 - Dec 1) from regular periods because Black Friday amplifies fraudulent signals.

The choice of replacing the missing device ids by artficially-created ids was made based on the fact that devices are mainly used by one user and transactions with missing device ids aren't more fraudulent than transactions with identified device ids.

## Overview of the Strongest Fraud Signals

#### Multi-entity behavior (5x-14x fraud multiplier)

The most powerful fraud indicators in this dataset is that fraudsters interact with more entities than legitimate users, especially on the Black Friday:   

| Behavior | Period | Fraud Rate | Baseline | Multiplier |
|---|---|---|---|---|
| User transacting at 2+ merchants | Black Friday | 77.9% | 5.4% | 14.4x |
| User transacting at 2+ merchants | Other weeks | 33.0% | 3.8% | 8.7x |
| Device linked to 2+ merchants | Black Friday | 83.7% | 6.4% | 13.1x |
| Device linked to 2+ merchants | Other weeks | 39.3% | 6.1% | 6.5x |
| User with 2+ cards | Black Friday | 43.0% | 4.4% | 9.8x |
| User with 2+ cards | Other weeks | 25.3% | 3.1% | 8.2x |
| Device with 2+ cards | Black Friday | 45.8% | 5.4% | 8.5x |
| Device with 2+ cards | Other weeks | 25.5% | 5.8% | 4.4x |
| User with 2+ devices | Black Friday | 38.7% | 5.4% | 7.2x |
| User with 2+ devices | Other weeks | 18.5% | 3.2% | 5.8x |

**What this means:** legitimate users in this dataset mostly stick to one card, one device, 
one merchant. Any deviation from that pattern is a strong fraud signal. 

**What led to this conclusion:** bivariate analysis grouping each entity by distinct related 
entities, then computing fraud rates after filtering out entities below a minimum transaction 
count (5 for merchants and BINs, 2 for users, cards, and devices) to avoid misleading rates 
from low-activity entities.

#### Transaction velocity

Transactions occurring within ≤1 minute of the previous transaction by the same user 
show a **52.6% fraud rate** (10/19) vs 12% baseline.    
The sample is small but the signal 
is consistent across Black Friday (50%) and non-Black Friday (55.6%) periods.

## Contextual Amplifiers

#### Transaction amount

High-value transactions carry elevated fraud rates in both periods, though with different magnitudes:

| Period | Above P95 (R$2.775) | Below P95 | Multiplier |
|---|---|---|---|
| Black Friday | 46.7% (90 txn) | 13.7% (1.955 txn) | 3.4x |
| Other weeks | 28.6% (70 txn) | 5.7% (1.084 txn) | 5.0x |

#### Black Friday period

The Black Friday window (Nov 22–Dec 1) doubles the overall fraud rate: **15.1% vs 7.1%** in other weeks.    
Nearly every fraud pattern intensifies during this period.    
For example, multi-merchant users during Black Friday reach 77.9% fraud rate (vs 32%), and middle of the night transactions fraud jump to 24.2% (vs 2.83%).

#### Middle of the night transactions (conditional)

Midnight-to-5AM is **not a standalone fraud signal**. Outside Black Friday, nighttime fraud (2.8%) 
is actually lower than daytime (7.5%).    
The elevated rate only appears during Black Friday: 
24.2% nighttime vs 13.9% other periods.    
Time-of-day should therefore only contribute to the risk score during high-activity periods.

#### Multi-card and multi-merchant velocity over time windows

The velocity window analysis reveals how fraud rates change depending on how quickly a user 
switches cards or merchants. The table below shows fraud rates for transactions where the user 
used multiple cards, multiple merchants, or both within a given time window:

| Window | Combo | BF fraud rate (txn) | Non-BF fraud rate (txn) |
|---|---|---|---|
| 60 min | multi-card only | 63.9% (72) | 36.4% (11) |
| 60 min | multi-merchant only | 83.3% (6) | - |
| 60 min | both | 66.7% (9) | - |
| 24h | multi-card only | 62.5% (128) | 33.3% (33) |
| 24h | multi-merchant only | 80.0% (5) | - |
| 24h | both | 81.0% (42) | 33.3% (3) |

During Black Friday, a user switching cards within 60 minutes shows a 63.9% fraud rate 
on 72 transactions: a strong enough signal with sufficient volume for a hard block. 
The combined signal (multiple cards AND multiple merchants within 24h) reaches 81.0% 
during Black Friday. Outside Black Friday, multi-card velocity still signals fraud 
at 33.3-36.4%, but on smaller samples, making it better suited for scoring than hard blocking.

#### High-fraud entities

Bivariate analysis identified specific entities with extreme fraud concentrations:

- **56 users** with 100% fraud rate (min 2 transactions)
- **67 cards** with 100% fraud rate (min 2 transactions)
- **49 devices** with 100% fraud rate (min 2 transactions)
- **11 merchants** with 100% fraud rate (min 5 transactions)

Users, cards, and devices with confirmed 100% fraud rates are candidates for blacklisting 
with periodic review.    
Merchants with extreme fraud concentration are treated differently: 
since merchants are CloudWalk's customers, these cases should trigger an investigation 
(compromised account? targeted by fraudsters?) and a remediation program rather than a hard block.

## Cross-Entity Graph Analysis

Beyond individual entity metrics, a graph-based analysis was performed to detect whether     
fraudulent users are **connected to each other** through shared cards or devices.

#### Method

For each card and device, all users who used it were identified.   
User pairs sharing an entity were treated as connected.   
Connected components (clusters) were extracted using a Union-Find algorithm.

#### Results

- **29 clusters** of 2+ users were found, all connected through shared cards (no shared devices were found)  
- Clustered users show **55.3% fraud rate** vs 10.2% for non-clustered users (5.4x)       
- 8 clusters have fraud rates ≥50%, while 21 clusters sit at exactly 0%, a clean separation between fraud rings and legitimate card sharing    
- **47 of 61 clustered users** were not flagged by any individual entity rule; the graph layer catches users that per-entity thresholds miss entirely

#### Case Study: Cluster 26 - Compromised credentials

User 91637 cycled through **20+ stolen cards** at a single merchant (4705) from a single device (563499),    
transacting exclusively between midnight and 3AM during Black Friday week. 
19 of 22 transactions received chargebacks.

The graph linked this user to 3 legitimate cardholders through a shared card (`496045******1160`), 
identifying **compromised credentials** invisible to per-entity rules.

#### Contrast: Cluster 5 - Legitimate Card Sharing

Three users shared one card across three different devices and three different merchants, with zero fraud.   
This is consistent with household card sharing: one card, each person on their own device, normal purchasing patterns.

| | Fraud cluster (26) | Legitimate cluster (5) |
|---|---|---|
| Cards per user | 20+ | 1 shared |
| Devices | 1 device, 1 user | 3 devices, 3 users |
| Merchants | 1 | 3 |
| Time pattern | Midnight–3AM bursts | Daytime, spaced out |
| Fraud rate | 76% | 0% |

**Fradulent behavior:** fraud funnels many cards into one device. Legitimate sharing spreads one card 
across many devices.

## Non-Signals

Three hypotheses tested negative and should **not** be used as detection features:

- **Missing device_id**: 8.1% fraud rate vs 13.7% for known devices. Missing device info 
  is not a fraud indicator in this dataset.
- **Microtransactions**: no meaningful correlation with fraud at user or card level, 
  even during Black Friday.
- **Weekday vs weekend**: identical fraud rates.

## Proposed Actions

Hard blocks are reserved for patterns where precision is inherently high (extreme fraud 
multipliers, low false-positive risk), protecting recall.    
Scoring handles moderate signals 
where outright blocking would reject too many legitimate transactions, protecting precision.

### Hard blocks (high confidence, low false-positive risk)

**Multi-entity velocity rules:**
- Block users using **2+ distinct cards within 60 minutes** during Black Friday 
  or promotional periods (63.9% fraud rate on 72 transactions during BF)
- Block users using **2+ cards AND 2+ merchants within 24 hours** regardless 
  of period (81.0% fraud rate during BF on 42 txn; 33.3% outside BF on 3 txn, 
  but the combined signal is strong enough to justify a hard block)
- Block users/devices transacting at **2+ merchants within a rolling 24h window** 
  (77.9% fraud rate during BF, 14.4x multiplier; 33.0% outside BF, 8.7x multiplier)
- Block transactions occurring **≤1 minute** after the previous by the same user 
  (50% during BF, 55.6% outside BF)

**Blacklist and history rules:**
- Block transactions from **blacklisted users, cards, or devices** (100% fraud rate, 
  min appearance threshold). Merchants are excluded from hard blacklists.
- Block users with a **previous chargeback** on record (retroactive: applied 
  once chargeback data is received, as specified in the case requirements)

### Score-based escalation (require step-up verification)

| Risk factor | Period context | Score | Rationale |
|---|---|---|---|
| User connected to fraud cluster | Both periods | +40 pts | 55.3% fraud rate for clustered users (5.4x) |
| User with 2+ cards within 60 min | Outside BF only | +25 pts | 36.4% fraud rate; strong but insufficient for hard block outside BF |
| User with 2+ devices | Both periods | +25 pts | BF: 38.7% (7.2x); non-BF: 18.5% (5.8x) |
| User with 2+ cards within 24h | Outside BF only | +15 pts | 33.3% fraud rate on 33 txn |
| Transaction amount above P95 (R$2.775) | Both periods | +15 pts | BF: 46.7% (3.4x); non-BF: 28.6% (5.0x) |
| Card shared across 2+ users | Both periods | +15 pts | BF: 32.8%; non-BF: 3.3% |
| Midnight-5AM | BF only (disabled otherwise) | +8 pts | BF: 24.2% vs 13.9%; non-BF: 2.8% vs 7.5% |
| BIN in high-risk tier (≥40% historical fraud rate) | Both periods | +8 pts | 8 BINs identified in 40-100% range |

### Graph-layer monitoring

- Flag any transaction from a user connected to a known fraud cluster 
  via shared entities (55.3% fraud rate for clustered users)
- Monitor cards appearing across multiple users for compromised credential 
  detection (as demonstrated with card `496045******1160` in cluster 26)
- Periodic cluster recomputation as new transaction data flows in

### Dynamic thresholds

- **Tighten rules during Black Friday and promotional periods**: multi-card 
  velocity within 60 minutes escalates from scoring to hard block; 
  scoring thresholds lower (e.g., deny threshold from 45 to 35 points)
- **Disable time-of-day rules outside promotional periods**: nighttime fraud 
  drops below daytime rates (2.8% vs 7.5%), so nighttime scoring adds 
  no value and would only generate false positives
- **Merchant remediation instead of blocking**: merchants with sustained 
  high chargeback rates enter remediation programs (mandatory 3D Secure, 
  lower transaction limits) rather than being blocked. Merchants failing 
  remediation may face suspension, but only after investigation confirms 
  the merchant is not simply a victim of targeted fraud
- Review and update entity blacklists on a rolling basis as fraud rates shift

# Additional Data Sources 

## MCC - Merchant category code

MCC codes would reveal whether fraud concentrates in specific industries (e.g., digital goods, gambling, electronics).   
It would also enable category-velocity rules: a user who normally transacts at grocery stores suddenly purchasing at a high-risk electronics 
merchant is a different signal than the same amount at a familiar category.

## Geolocation/IP address

- Distance-based velocity checks (two transactions from different cities within minutes)
- VPN/proxy detection (common in fraud to mask true origin)
- Country mismatch between card issuer (BIN) and transaction origin

## Device fingerprint details

Device data: OS, browser, screen resolution, language settings, whether the device is rooted/jailbroken, would enable:
- Detection of emulators or virtual machines (common in bot-driven fraud)
- Identification of device spoofing (same physical device presenting multiple fingerprints)
- Risk scoring based on device age (first seen vs established)

## User historical spending profile

- Spending deviation detection (current transaction vs user's average/median)
- Category deviation (new merchant type for this user)
- Frequency deviation (user who transacts once a month suddenly making 5 transactions in a night)

## Chargeback timing and reason codes

Knowing the chargeback reason code (e.g., "unauthorized transaction" vs "product not received" vs "duplicate charge")    
would separate **true fraud** from **friendly fraud** (cardholder received the goods 
but disputes anyway).    
These require very different responses: true fraud warrants 
blocking the card, friendly fraud warrants merchant-side controls.

The time gap between transaction and chargeback filing would also reveal how quickly 
fraud is detected by cardholders, informing monitoring window lengths.

## 3D Secure/authentication method

Whether the transaction went through cardholder authentication (e.g., 3D Secure, OTP verification),   
because authenticated transactions have lower fraud rates by design, and the absence of authentication on a high-value transaction is an additional risk factor.

## Card issuance and lifecycle data

- Card age (recently issued cards are higher risk)
- Whether the card was recently reported lost/stolen
- Whether there have been recent PIN/password resets on the associated account

## Merchant risk profile

- Merchant age on the platform (new merchants carry higher risk)
- Merchant chargeback-to-transaction ratio over time
- Whether the merchant has been flagged by card networks

## Data for graph-layer enrichment

- **Shared email addresses or phone numbers**: across accounts
- **Shared billing/shipping addresses**: multiple users sending to the same address
- **Shared payment credentials**: beyond card number (e.g., same bank account for refunds)
- **Session-level data**: users logging in from the same IP or session token

# Recommendations

## Preventive Controls (real-time, pre-authorization)

These rules execute at transaction time and either block or escalate before approval.   

Hard blocks prioritize recall where precision is inherently high, while scoring preserves precision for moderate signals.

### Hard blocks

**Velocity rules:**
- Reject if the user has made a transaction within the last 1 minute 
  (BF: 50% fraud rate; non-BF: 55.6%)
- Reject if the user has exceeded N transactions within a rolling time window 
  (calibrate N based on production data; the dataset shows legitimate users rarely 
  exceed 2-3 transactions per day)

**Multi-entity rules:**
- Reject if the user has transacted at 2+ distinct merchants within a rolling 24h window 
  (BF: 77.9% fraud rate, 14.4x multiplier; non-BF: 33.0%, 8.7x)
- Reject if the device has been associated with 2+ distinct merchants within a rolling 24h window 
  (BF: 83.7%, 13.1x; non-BF: 39.3%, 6.5x)
- Reject if the user has used 2+ distinct cards within 60 minutes during Black Friday 
  or promotional periods (BF: 63.9% fraud rate on 72 transactions)
- Reject if the user has used 2+ cards AND 2+ merchants within 24 hours 
  (BF: 81.0% on 42 txn; non-BF: 33.3% on 3 txn)

**Blacklist enforcement:**
- Reject if user_id, device_id, or card_number appears on the active hard blacklist 
  (maintained from entities with historically confirmed fraud).   
  Merchants are excluded from hard blacklists since they are CloudWalk's direct customers; 
  high-chargeback merchants are handled through remediation.

**Chargeback history:**
- Reject if the user had a previous chargeback 
  (as specified in the case requirements; chargeback data arrives days after approval, 
  so this rule applies retroactively once the flag is received)

### Score-based escalation

Suspicious patterns that are not strong enough for a hard block trigger score accumulation.    
When the cumulative score exceeds a defined threshold, the transaction is held for 
step-up authentication (e.g., OTP, identity confirmation) rather than rejected outright.

| Risk factor | Period context | Score | Rationale |
|---|---|---|---|
| User with 2+ cards within 60 min | Outside BF only | +25 pts | 36.4% fraud rate (11 txn); strong signal but insufficient volume for hard block outside BF |
| User with 2+ cards within 24h | Outside BF only | +15 pts | 33.3% fraud rate (33 txn) |
| User with 2+ devices | Both periods | +25 pts | BF: 38.7% (7.2x); non-BF: 18.5% (5.8x) |
| Transaction amount > P95 (R$2.775) | Both periods | +15 pts | BF: 46.7% (3.4x); non-BF: 28.6% (5.0x) |
| Card shared by 2+ users | Both periods | +15 pts | BF: 32.8%; non-BF: 3.3% |
| Midnight-5AM | BF only (disabled otherwise) | +8 pts | BF: 24.2% vs 13.9%; non-BF: 2.8% vs 7.5% (inverted signal) |
| BIN in high-risk tier (≥40% historical fraud rate) | Both periods | +8 pts | 8 BINs identified in the 40-100% range |
| User connected to fraud cluster | Both periods | +40 pts | 55.3% fraud rate for clustered users (5.4x) |


## Detective Controls (post-authorization monitoring)

### Entity monitoring
- Track fraud rate per merchant, BIN, device, and user on rolling windows 
  (e.g., 7-day and 30-day)
- Trigger alerts when any entity crosses predefined fraud rate thresholds 
  (e.g., merchant exceeding 10% chargeback rate over 50+ transactions)
- Feed alerts into a review queue for manual investigation
- Apply **tighter thresholds during Black Friday and promotional periods**, 
  where baseline fraud rates double (15.1% vs 7.1%)

### Graph recomputation
- Periodically rebuild the entity graph (user ↔ card ↔ device) and recompute 
  connected components
- Flag new clusters forming around known fraudulent entities
- Identify compromised cards early: if a card appears in a cluster with a 
  confirmed fraudster, flag it for monitoring even if its individual transactions 
  look clean (as demonstrated with card `496045******1160` in cluster 26)

### Seasonal threshold adjustment
- Tighten scoring thresholds during Black Friday and promotional periods: 
  multi-card velocity within 60 minutes escalates from scoring to hard block; 
  deny threshold lowers (e.g., from 45 to 35 points)
- Disable nighttime scoring outside high-activity periods, 
  where nighttime fraud (2.8%) is below daytime rates (7.5%)

## Reactive Processes (post-chargeback)

### Immediate response
- Upon receiving a chargeback, retroactively flag the user, card, and device 
  for enhanced monitoring
- Cross-reference the flagged card against the entity graph to identify other 
  users or devices potentially compromised by the same actor
- If the merchant's chargeback rate exceeds a threshold after the new case, 
  escalate for merchant review (possible compromise or targeted attack)


### Blacklist management
- Add confirmed fraud entities to blacklists with expiration policies 
  (cards and devices may be recycled or legitimately change hands over time)
- Maintain tiered blacklists: hard block (confirmed fraud, users/cards/devices only) 
  vs watch list (elevated monitoring, score boost)
- Review blacklists periodically to avoid stale entries blocking legitimate users

### Merchant accountability
- Merchants with sustained high chargeback rates enter remediation programs 
  rather than being blocked. Since merchants are CloudWalk's customers, the goal 
  is to help them reduce fraud exposure, not penalize them for being targeted.
- Remediation measures: mandatory 3D Secure enrollment, lower transaction limits, 
  enhanced monitoring of incoming transactions
- In the dataset, 11 merchants showed 100% fraud rates (min 5 transactions). 
  These require investigation first: a merchant with 100% chargebacks could be 
  a compromised account, a target of coordinated fraud, or in rare cases a 
  deliberately fraudulent operation. The response differs for each scenario.
- Merchants failing remediation after investigation may face suspension as a 
  last resort

## Balancing fraud prevention with customer experience

The detection approach is designed to maximize fraud catch rate (recall) 
without generating excessive false positives that would hurt merchant sales:

- **Hard blocks** are reserved for patterns with extreme fraud multipliers (≥7x in both periods) 
  where precision is inherently high. These protect merchants from guaranteed chargeback losses 
  on transactions that are almost certainly fraudulent.
- **Score-based escalation** adds friction (step-up verification) rather than 
  immediate rejection for moderate signals. The legitimate customer can still complete 
  the transaction after verification, preserving the sale for the merchant.
- **Seasonal adjustment** avoids applying Black Friday-calibrated rules to 
  normal periods, where they would block legitimate transactions unnecessarily. 
  Conversely, it tightens rules during promotional periods when the cost of 
  missed fraud is highest.
- **Non-signals are excluded**: missing device_id, microtransactions, and 
  weekday/weekend patterns were tested and deliberately left out of the 
  rule set. Including them would increase rule complexity and false positive 
  risk for zero recall gain.
- **Merchants are never hard-blocked**: detection rules target user-side and 
  device-side behavior. Merchants with fraud problems receive remediation 
  support, not punishment.

# Anti-fraud solution

## Architecture Overview - Rule-based system

The proposed solution is a **three-layer rule-based detection system** that evaluates 
every incoming transaction in real time and returns one of three decisions: 
**approve**, **deny**, or **review** (escalate to manual analysis).

The system is designed around the assumption that
false negatives (missed fraud leading to chargebacks) are costlier than false positives 
(blocked legitimate transactions). Hard blocks target high-multiplier patterns where 
precision is inherently high. Scoring handles moderate signals where outright blocking 
would hurt merchant sales.

The layers execute sequentially, each adding context to the decision:

**Layer 1 - Transactional (per-transaction rules)**
Evaluates the transaction in isolation: amount, timestamp, blacklist membership.
Target latency: <10ms. These are simple lookups and threshold comparisons.

**Layer 2 - Entity (behavioral rules)**
Evaluates the transaction against the entity's recent history: velocity, 
multi-entity patterns, chargeback history. Requires a state store holding 
rolling aggregates per user, card, device, and merchant.
Target latency: <30ms.

**Layer 3 - Relational (graph-based rules)**
Evaluates whether the transacting entities are connected to known fraud 
clusters through shared cards or devices. Requires a precomputed entity graph 
updated periodically (batch, not real-time).
Target latency: <50ms (graph lookup, not recomputation).

Total target latency: **<50ms end-to-end** for the rule engine decision, 
well within typical payment authorization windows.

#### Data flow

1. Transaction payload arrives (merchant_id, user_id, card_number, device_id, 
   transaction_amount, transaction_date)
2. **Enrichment** - payload is joined with:
   - Blacklists (user, card, device, BIN)
   - Rolling aggregates from the state store (transaction count, distinct 
     entity counts, last transaction timestamp per user/card/device)
   - Graph cluster membership from the precomputed entity graph
   - Current seasonal period flag (Black Friday / promotional vs normal)
3. **Rule evaluation** - enriched transaction passes through Layer 1 → 2 → 3.
   Each layer can short-circuit: if Layer 1 issues a hard deny, Layers 2 and 3 
   are skipped.
4. **Decision** - the cumulative risk score determines the outcome:
   - Score below review threshold → **approve**
   - Score between review and deny thresholds → **review**
   - Score above deny threshold → **deny**
   - Any hard-block rule triggered → **deny** (overrides score)
5. **Logging** - every decision is logged with the full rule evaluation trace 
   (which rules fired, individual score contributions, final score). 
   This is essential for auditability, dispute resolution, and rule tuning.
6. **Feedback loop** - when chargeback data arrives (days later), the 
   corresponding transaction is labeled as fraud. This feeds into:
   - Blacklist updates (flagged entities added)
   - Entity graph recomputation (new fraud edges)
   - Rule performance monitoring (did the rules catch it?)


## Rule Engine Design

### Decision types

The system uses two types of rules, following industry-standard rule engine design:

**Logic conditions (hard blocks)** - binary rules that trigger an immediate deny. 
Used for patterns where the fraud multiplier is high enough (≥7x in both periods) 
that the false positive rate is inherently low.   
These are "if X, then deny" with no scoring involved.

**Risk scoring rules** - each rule contributes a weighted score to the transaction. 
The cumulative score determines the final decision. This approach reduces false 
positives compared to pure logic conditions, because moderate signals that would 
individually cause false positives can be tolerated unless they co-occur.

### Scoring mechanism

Each risk scoring rule assigns points based on the severity of the signal. 
The weights are calibrated from the fraud multipliers observed in the analysis, 
using the **non-Black Friday multiplier as the base** (since these represent 
the signal's strength under normal conditions). The seasonal modifier (S1) 
handles the amplification during promotional periods.

| Weight tier | Fraud multiplier (non-BF base) | Points | Example |
|---|---|---|---|
| Critical | ≥10x | 40 | Graph cluster member (5.4x overall, but catches 47 users missed by other rules) |
| High | 5x-10x | 25 | User with 2+ devices (5.8x non-BF), multi-card within 60min (scored outside BF) |
| Medium | 3x-5x | 15 | Amount > P95 (5.0x non-BF), card shared by 2+ users |
| Low | 1.5x-3x | 8 | Nighttime during BF (1.7x BF-only), high-risk BIN |

Decision thresholds (starting point, to be calibrated in production):
- **Score < 20** → approve
- **Score 20-45** → review (step-up verification or manual queue)
- **Score > 45** → deny

These thresholds are configurable per merchant segment and per seasonal period. 
During Black Friday, the deny threshold should be tightened (e.g., from 45 to 35) 
to reflect the elevated fraud environment where nearly every signal intensifies.


## Rule Catalog

### Layer 1 - Transactional rules

| Rule ID | Type | Condition | Action |
|---|---|---|---|
| T1 | Hard block | user_id, card_number, or device_id on hard blacklist | Deny |
| T2 | Hard block | User has previous chargeback on record | Deny |
| T3 | Scoring | transaction_amount > P95 (R\$2.775) | +15 pts |
| T4 | Scoring | transaction_amount > P99 (R\$4.028) | +25 pts |
| T5 | Scoring | BIN in high-risk tier (≥40% historical fraud rate) | +8 pts |
| T6 | Scoring | Midnight-5AM AND high-activity period (Black Friday) | +8 pts |

Merchants are excluded from T1 because they are CloudWalk's direct customers. 
Merchants with high chargeback rates are handled through remediation programs 
(see section 4.4), not real-time blocking.

T6 is disabled outside promotional periods, since nighttime 
fraud outside Black Friday (2.8%) is below daytime rates (7.5%).

### Layer 2 - Entity rules

| Rule ID | Type | Condition | Action |
|---|---|---|---|
| E1 | Hard block | User transacted at 2+ distinct merchants in rolling 24h | Deny |
| E2 | Hard block | Device linked to 2+ distinct merchants in rolling 24h | Deny |
| E3 | Hard block | Time since user's last transaction ≤ 1 minute | Deny |
| E4 | Conditional | User with 2+ distinct cards within 60 min | Deny during BF; +25 pts outside BF |
| E5 | Hard block | User with 2+ cards AND 2+ merchants within 24h | Deny |
| E6 | Scoring | User associated with 2+ distinct devices in rolling 7 days | +25 pts |
| E7 | Scoring | Card used by 2+ distinct users in rolling 7 days | +15 pts |
| E8 | Scoring | User transaction count > N in rolling 24h | +15 pts |
| E9 | Scoring | Device associated with 2+ distinct BINs in rolling 7 days | +15 pts |

Data support for hard blocks (BF / non-BF):
- E1: 77.9% / 33.0% fraud rate (14.4x / 8.7x multiplier)
- E2: 83.7% / 39.3% (13.1x / 6.5x)
- E3: 50% / 55.6% (consistent across periods)
- E4: 63.9% on 72 txn during BF / 36.4% on 11 txn outside BF (hard block justified during BF by volume and rate; scored outside BF due to small sample)
- E5: 81.0% on 42 txn during BF / 33.3% on 3 txn outside BF (combined signal strong enough for hard block in both periods)

### Layer 3 - Relational rules

| Rule ID | Type | Condition | Action |
|---|---|---|---|
| G1 | Scoring | User belongs to a cluster with ≥50% fraud rate | +40 pts |
| G2 | Scoring | Card appeared in a cluster containing a confirmed fraudster | +25 pts |
| G3 | Scoring | User shares a card with a user who has a chargeback | +25 pts |

The graph was built across the full dataset (both periods), since entity 
relationships span time windows. Clustered users show 55.3% fraud rate 
vs 10.2% for non-clustered users. 47 of 61 clustered users were not 
flagged by any Layer 1 or Layer 2 rule, making this layer essential for 
coverage.

### Seasonal modifier

| Modifier | Condition | Effect |
|---|---|---|
| S1 | Transaction falls within Black Friday / promotional period | All scoring rules receive 1.5x multiplier; deny threshold tightens (e.g., 45 → 35) |
| S2 | Transaction falls outside promotional periods AND midnight-5AM | Nighttime rule T6 is disabled |
| S3 | Transaction falls within Black Friday / promotional period | Rule E4 escalates from scoring (+25 pts) to hard block (deny) |

## Operational Considerations

### State store

Layer 2 rules require rolling aggregates per entity. The state store must support 
fast writes (update counters on every transaction), fast reads (lookup aggregates 
during enrichment), and time-windowed expiration (60min, 24h, and 7-day windows).

### Entity graph

Layer 3 requires a precomputed graph of entity relationships. Since recomputation 
is expensive and fraud rings don't form in milliseconds, this runs as a 
**batch process** (e.g., hourly or daily):

1. Scan recent transactions for shared cards and devices
2. Build user-to-user edges
3. Compute connected components (Union-Find)
4. Calculate per-cluster fraud rates using chargeback labels
5. Export cluster membership and fraud rates to a lookup table 
   accessible by the real-time rule engine

### Blacklist management

Blacklists are maintained as lookup tables with two tiers:

**Hard blacklist** - confirmed fraud entities (users, cards, devices only; 
merchants are excluded). Triggers immediate deny via rule T1.

**Watch list** - elevated-risk entities (above threshold but not confirmed). 
Adds score points rather than blocking.

Entries have expiration dates to prevent stale blocks. Cards and devices 
may be recycled legitimately over time. Periodic review ensures the blacklist 
reflects current risk, not historical noise.

### Merchant remediation

1. Alert triggers when a merchant's chargeback rate exceeds a threshold 
   (e.g., 10% over 50+ transactions on a rolling 30-day window)
2. Investigation determines the cause: compromised account, targeted by 
   fraudsters, or operational issues
3. Remediation measures are applied based on the finding: mandatory 3D Secure, 
   lower per-transaction limits, enhanced monitoring of incoming transactions
4. Merchants failing remediation after a defined period may face suspension 
   as a last resort

### Rule lifecycle

Rules degrade over time as fraud tactics evolve. The system must include:

**Performance monitoring** - track per-rule metrics continuously:
true positive rate (how often the rule correctly flags fraud), 
false positive rate (how often it flags legitimate transactions), 
coverage (what percentage of total fraud this rule catches), 
and redundancy (whether another rule already catches the same fraud).

**Testing of new rules** - proposed rule changes are tested in shadow 
mode against live traffic. The new rule evaluates transactions and logs 
what it would have decided, but does not affect the actual decision. 
Performance is compared against the current production rule before rollout.

**Seasonal recalibration** - thresholds and weights are reviewed before 
known high-activity periods (Black Friday, holiday season) based on 
prior-year patterns.

**Rule retirement** - rules with sustained low true-positive rates or 
that duplicate other rules are candidates for removal. This prevents 
rule sprawl: an ever-growing rule set where rules contradict or 
neutralize each other, degrading overall performance.

### Decision transparency

Every decision produces an audit trail containing:
- The full enriched transaction payload
- Which rules fired (rule IDs)
- Individual score contributions
- Final cumulative score
- Decision outcome (approve/review/deny)
- Timestamp and processing latency

This serves three purposes: regulatory compliance (auditors can inspect 
decision logic), dispute handling (customer support can explain why a 
transaction was declined), and rule tuning (analysts can trace false 
positives back to the offending rule).

## Limitations and Evolution Path

### Current limitations of a pure rule-based approach

**Known-pattern dependency** - rules can only catch fraud patterns that have been 
explicitly identified and encoded. Novel fraud tactics that don't trigger any 
existing rule will pass through undetected until the pattern is recognized and 
a new rule is written.

**Manual maintenance** - adding, tuning, and retiring rules requires human effort. 
As transaction volume grows and fraud tactics evolve, the rule catalog needs 
continuous attention. Without disciplined lifecycle management, rule sprawl 
becomes a real risk.

**Threshold rigidity** - while scoring provides more flexibility than pure logic 
conditions, the weights and thresholds are still static between calibration cycles. 
A fraud pattern that gradually shifts (e.g., amount distribution changes) may 
erode a rule's effectiveness before the next review.

### Hybrid evolution (rules + ML)

A natural evolution path is a **hybrid system** where the rule engine handles 
known patterns with full transparency, and a machine learning layer handles 
anomaly detection for unknown or evolving patterns:

- **Rules** remain the first line of defense for hard-block conditions 
  and well-understood risk factors. They provide the explainability 
  and auditability required for compliance.
- **An ML scoring model** runs in parallel, 
  producing a probability score based on all available features. 
  This score feeds into the rule engine as an additional input - 
  treated like any other risk factor with its own weight.
- **The ML model detects what rules miss** - subtle behavioral anomalies, 
  feature interactions that no single rule captures, and emerging 
  patterns before they're codified as rules.
- **Rules can be generated from ML insights** - when the model consistently 
  flags a pattern, analysts can extract it and encode it as an explicit 
  rule, making the detection transparent and reducing reliance on the 
  model for that specific pattern.

This hybrid approach preserves the transparency and auditability of the 
rule-based system while gaining the adaptability of machine learning - 
addressing the main vulnerability of a purely rule-based approach without 
sacrificing explainability.

## Rule Coverage Validation

To assess whether the rule catalog from section 4.3 behaves as expected, a chronological simulation was run against the same dataset used to derive the rules. All 3.199 transactions were replayed in timestamp order, maintaining rolling state per entity (user, card, device) so that each transaction was evaluated only against history available at the time it occurred.

**This is not a predictive performance benchmark.** The rules were derived from this dataset, so testing them on the same data is circular. The results measure rule coverage and false positive exposure, not how the system would perform on unseen transactions. 100% recall on deny + review is expected under these conditions and should not be interpreted as a generalization metric.

The value of this exercise lies in three areas: identifying which rules carry the most weight, estimating how many legitimate transactions would be blocked, and exposing redundancy or weakness in individual rules.

### Simulation design

The simulation replays transactions chronologically and evaluates every rule in the catalog against the state available at each transaction's timestamp. The key design choices:

**Rolling state store** - for each transaction, the simulator maintains per-entity history (user, card, device) with time-windowed lookups (1 minute, 60 minutes, 24 hours, 7 days). Entity rules (Layer 2) query this state to compute velocity, multi-card usage, multi-merchant counts, and rapid-fire detection. The current transaction is evaluated *before* being added to state, so it cannot appear in its own history.

**Chargeback delay** - rule T2 (previous chargeback) simulates a 3-day processing delay. A user's first fraudulent transaction cannot be caught by T2 because the chargeback has not arrived yet. Only subsequent transactions after the delay window get flagged. This is the only temporal rule where the simulation genuinely prevents look-ahead bias.

**Static lookups (circular)** - blacklists (T1), high-risk BINs (T5), and graph clusters (G1, G2, G3) are pre-computed from the full dataset including labels. This means these rules have access to information that would not exist at transaction time in production. The blacklist in particular was built by identifying entities with 100% fraud rate across the entire dataset; in reality, a blacklist only catches repeat offenders *after* their first fraud event.

**Scoring and decisions** - each transaction accumulates a risk score from all firing rules. Hard-block rules (T1, T2, E1-E3, E5, and E4 during Black Friday) trigger an immediate deny regardless of score. For scored transactions, the thresholds are: score < 20 = approve; 20-45 = review; > 45 = deny. During Black Friday, the deny threshold tightens to 35 and all scoring rules receive a 1.5x multiplier (S1).

### System-level results

| Metric | Value |
|---|---|
| Total transactions | 3.199 |
| Total fraud | 391 (12.2%) |
| Total legitimate | 2.808 |

**Decision distribution:**

| Decision | Total | Fraud | Legitimate |
|---|---|---|---|
| Approve | 2.568 | 0 | 2.568 |
| Review | 145 | 21 | 124 |
| Deny | 486 | 370 | 116 |

**Deny only:** recall 0.946 (370/391), precision 0.761 (370/486), F2-score 0.902.

**Deny + review (all flagged):** recall 1.000 (391/391), precision 0.620 (391/631), false positive rate 8.5% (240/2.808).

Zero fraud transactions received an "approve" decision. As noted above, this is expected given the circularity and should not be presented as a performance claim.

**By period:**

| Period | Transactions | Fraud | Denied | TP | FP | Recall | Precision |
|---|---|---|---|---|---|---|---|
| Black Friday | 2.045 | 309 | 409 | 309 | 100 | 1.000 | 0.756 |
| Other weeks | 1.154 | 82 | 77 | 61 | 16 | 0.744 | 0.792 |

The non-Black Friday recall of 74.4% is the most informative number in these results. The Black Friday period benefits from aggressive rule tuning (S1 multiplier, S3 hard block on multi-card velocity, T6 nighttime scoring, lowered deny threshold), which inflates its recall. Outside Black Friday, where fewer rules apply and thresholds are relaxed, 21 fraud transactions survive to "review" rather than "deny." This gap reflects the system's period dependency: it was calibrated primarily around Black Friday patterns, and its coverage drops when those amplifiers are absent.

### Per-rule diagnostics

Each rule is measured on two axes: precision (what fraction of its triggers are actual fraud) and coverage (what fraction of total fraud it catches). Rules are sorted by coverage.

| Rule | Layer | Fired | TP | FP | Precision | Coverage |
|---|---|---|---|---|---|---|
| T1 - Blacklist | L1 | 232 | 232 | 0 | 100.0% | 59.3% |
| G3 - Shares card with fraudster | L3 | 286 | 216 | 70 | 75.5% | 55.2% |
| T5 - High-risk BIN | L1 | 293 | 161 | 132 | 54.9% | 41.2% |
| E9 - Device with 2+ BINs (7d) | L2 | 189 | 116 | 73 | 61.4% | 29.7% |
| E6 - User with 2+ devices (7d) | L2 | 161 | 88 | 73 | 54.7% | 22.5% |
| E8 - User high velocity (24h) | L2 | 102 | 81 | 21 | 79.4% | 20.7% |
| G2 - Card in fraud cluster | L3 | 91 | 78 | 13 | 85.7% | 19.9% |
| G1 - User in fraud cluster | L3 | 91 | 78 | 13 | 85.7% | 19.9% |
| E4 - User with 2+ cards (60min) | L2 | 97 | 67 | 30 | 69.1% | 17.1% |
| T3 - Amount > P95 | L1 | 160 | 62 | 98 | 38.8% | 15.9% |
| T2 - Previous chargeback | L1 | 72 | 58 | 14 | 80.6% | 14.8% |
| T6 - Night + BF | L1 | 236 | 57 | 179 | 24.2% | 14.6% |
| E1 - User at 2+ merchants (24h) | L2 | 58 | 51 | 7 | 87.9% | 13.0% |
| E2 - Device at 2+ merchants (24h) | L2 | 53 | 47 | 6 | 88.7% | 12.0% |
| E5 - User multi-card + merchant (24h) | L2 | 52 | 46 | 6 | 88.5% | 11.8% |
| E7 - Card with 2+ users (7d) | L2 | 25 | 14 | 11 | 56.0% | 3.6% |
| T4 - Amount > P99 | L1 | 31 | 14 | 17 | 45.2% | 3.6% |
| E3 - Rapid-fire (<=1 min) | L2 | 18 | 9 | 9 | 50.0% | 2.3% |

**Rule overlap:** most fraud transactions trigger multiple rules. 19.2% of fraud is caught by exactly 1 rule, 21.7% by 2 rules, 18.2% by 3 rules, and the remaining 40.9% by 4 or more. The most common single-rule catches come from T1 (blacklist), which dominates the catalog.

### Strengths identified

**Layered coverage works.** No single layer catches all fraud alone. Layer 1 (transactional) catches the most through T1, but Layers 2 and 3 contribute substantially: E8, E9, and G3 each cover 20-55% of fraud that would otherwise depend on blacklists alone. The graph layer (G1, G2, G3) is particularly important because 47 clustered users were not flagged by any Layer 1 or Layer 2 rule.

**Hard-block precision is high.** Rules E1, E2, and E5 (multi-merchant and combined multi-card/merchant velocity) all exceed 85% precision while maintaining reasonable coverage. These are the safest hard blocks in the catalog: they rarely fire on legitimate transactions and catch real fraud when they do.

**The scoring mechanism absorbs moderate signals without over-blocking.** 145 transactions landed in "review" rather than "deny," including 21 fraud transactions. This means the score thresholds successfully separated high-confidence fraud (deny) from ambiguous cases (review) for a large portion of transactions. In production, the review queue would allow manual analysts to resolve these without automatically blocking legitimate customers.

### Weaknesses identified

**T1 (blacklist) dominance and circularity.** T1 alone covers 59.3% of fraud with perfect precision, but this is the most inflated metric in the simulation. In production, a blacklist only catches entities *after* their first confirmed fraud event. On a 30-day dataset, many of these entities would not yet be blacklisted at the time of their first fraudulent transaction. The system's real-world coverage without T1 would be substantially lower.

**G1 and G2 are perfectly redundant.** Both rules fire on exactly the same 91 transactions (78 TP, 13 FP). G1 checks whether the user belongs to a high-fraud cluster; G2 checks whether the card appears in a cluster with a confirmed fraudster. In this dataset, these conditions are identical. One of the two could be removed or merged without losing any coverage, simplifying the rule catalog.

**T6 (nighttime + BF) has the worst precision in the catalog.** It fires on 236 transactions but only 57 are fraud: 24.2% precision. That means 179 legitimate transactions are flagged just for occurring between midnight and 5AM during Black Friday. At +8 points, T6 alone does not trigger a deny, but combined with even one other moderate signal (T5 high-risk BIN at +8, or T3 amount > P95 at +15), it pushes transactions into the review or deny zone. During the highest-revenue period of the year, this is a meaningful source of false positives.

**T3 and T4 (amount thresholds) have low precision.** T3 fires on 160 transactions with only 38.8% precision (98 false positives), and T4 on 31 transactions at 45.2% precision. High-value transactions are a contextual amplifier, not a standalone fraud signal. These rules are correctly implemented as scoring rather than hard blocks, but their contribution to false positives is substantial when stacked with other scoring rules during Black Friday (where S1 applies the 1.5x multiplier).

**Non-BF coverage drops significantly.** Outside Black Friday, recall falls to 74.4%. The 21 fraud transactions that escape to "review" instead of "deny" are cases where scoring rules accumulate enough points for review but not enough for a deny under the relaxed threshold (45 vs 35 during BF). This suggests the non-BF rule set may need additional signals or tighter calibration for normal periods.

**E3 (rapid-fire) has negligible impact.** Only 18 transactions trigger this rule, catching 9 fraud and 9 legitimate. The 1-minute window may be too tight for this dataset, or rapid-fire attacks are simply rare. The rule is conceptually sound (transaction velocity is a real fraud signal in production) but contributes almost nothing here.

### False positive impact

The simulation blocked 116 legitimate transactions outright (deny) and flagged another 124 for review, totaling 240 legitimate transactions affected out of 2.808 (8.5%).

During Black Friday specifically, 100 legitimate transactions were denied out of 1.736 legitimate BF transactions (5.8% block rate). Multiplied by the dataset's mean transaction amount, that would represent roughly R\$76.781 in potentially lost sales during the highest-revenue window. In practice, some of these would be recovered through the review queue (step-up verification), so the actual revenue impact depends on how quickly the review process resolves.

The largest contributors to false positives among scoring rules are T6 (179 FP), T5 (132 FP), T3 (98 FP), E6 (73 FP), E9 (73 FP), and G3 (70 FP). T6 and T3 are the most actionable: reducing T6's weight or tightening T3's threshold (from P95 to a higher percentile) would have the largest impact on false positive reduction with the smallest coverage loss.

### Actionable next steps

**Rule catalog cleanup:**
- Merge or remove G2 (redundant with G1 on this dataset). If future data shows divergence between cluster membership and card-level fraud presence, reintroduce G2 as a separate signal.
- Evaluate T6 for removal or weight reduction. Its 24.2% precision generates the most false positives of any rule, and its coverage (14.6%) overlaps heavily with other rules that fire on the same Black Friday fraud transactions.
- Monitor E3 in production; if rapid-fire attacks remain rare, consider widening the window (e.g., 5 minutes instead of 1 minute) or removing the rule.

**Threshold calibration:**
- The non-BF deny threshold (45 points) leaves 21 fraud transactions in review. Lowering it to 40 would catch some of these but at the cost of more false positives. The right threshold depends on the review team's capacity and the cost ratio between missed fraud and blocked sales; this should be calibrated with production data.
- T3's P95 threshold (R$2.775) is broad. Raising it to P97 or P98 would reduce the 98 false positives while retaining most coverage, since the fraud-amount distribution skews higher than the legitimate one.

**Validation on unseen data:**
- The most important next step is testing on data the rules have never seen. With only 30 days of data and no holdout set, a temporal split (e.g., train on days 1-20, test on days 21-30) would be too small to produce reliable estimates. The proper validation requires a new batch of transactions where blacklists, BIN risk tiers, and graph clusters are built incrementally as chargebacks arrive, not from the full labeled dataset.
- Shadow mode deployment (logging rule decisions without acting on them) against live traffic would provide the first honest performance metrics, especially for false positive rates.

**Monitoring infrastructure:**
- Track per-rule precision and coverage over time. Rules that degrade (precision drops, coverage falls) are candidates for recalibration or retirement.
- Track the review queue volume. If review transactions grow disproportionately to deny transactions, the scoring weights or thresholds may need adjustment.

# Results

## Main Findings Summary

Analysis of 3.199 CNP transactions (12.22% fraud rate) revealed that fraud in this 
dataset is driven primarily by **fraudsters interacting with more entities** (cards, 
devices, merchants) than legitimate users. Legitimate users mostly stick to one card, 
one device, one merchant; any deviation from that pattern is a strong fraud signal, 
especially during Black Friday.

All findings are split by period because general statistics mixing Black Friday 
(15.1% fraud rate) with other weeks (7.1%) are misleading. Black Friday doubles 
the baseline and amplifies every signal.

The strongest signals:

| Pattern | Black Friday | Other weeks | Layer |
|---|---|---|---|
| Device at 2+ merchants (24h) | 83.7% (13.1x) | 39.3% (6.5x) | Entity |
| User with 2+ cards + 2+ merchants (24h) | 81.0% (42 txn) | 33.3% (3 txn) | Entity |
| User at 2+ merchants (24h) | 77.9% (14.4x) | 33.0% (8.7x) | Entity |
| User with 2+ cards (60 min) | 63.9% (72 txn) | 36.4% (11 txn) | Entity |
| User in fraud-linked cluster | 55.3% (full dataset, 5.4x) | - | Relational |
| Rapid-fire transactions (≤1 min) | 50.0% (10 txn) | 55.6% (9 txn) | Entity |
| Amount above P95 (R$2.775) | 46.7% (3.4x) | 28.6% (5.0x) | Transactional |
| User with 2+ cards (7d) | 43.0% (9.8x) | 25.3% (8.2x) | Entity |

Three hypotheses tested **negative** and were excluded from the rule set: 
missing device_id, microtransactions, and weekday vs weekend. Including these 
would increase false positives for zero recall gain.

## Reasoning

The analysis followed several stages of increasing depth:

**Data description** - basic understanding of the dataset.

**Feature engineering** - missing values treatment and entity decomposition for better understanding.

**Univariate analysis** - each entity was analyzed individually to extract informations such as unique and mean values. 

**Bivariate analysis** - each entity was compared with the target variable to understand how each of them are related to fraud cases.

**Outlier analysis** - analysis of above the average behaviors that could influentiate results across several hypothesis. 

**Hypothesis validation** - 22 hypotheses were tested from a business 
understanding mind map and tested against the data, computed with minimum appearance thresholds to 
filter noise, always split by Black Friday 
vs non-Black Friday periods. This identified which entities concentrate fraud in specific condititions and which 
behavioral dimensions (multi-merchant, multi-card, multi-device) separate 
fraud from legitimate activity. 

**Cross-entity graph analysis** - users were connected through shared cards 
and devices using a Union-Find algorithm to detect clusters. 29 clusters 
were found, splitting cleanly into fraud clusters (≥50% fraud rate) and 
legitimate card-sharing clusters (0% fraud rate). This layer identified 
47 users that no individual entity rule would have caught.

**Case-level inspection** - the largest fraud cluster was examined 
transaction by transaction, revealing a card enumeration attack: one user 
cycling 20+ stolen cards through a single merchant and device during 
nighttime hours across Black Friday week. Contrast with a legitimate 
cluster showed the distinguishing signature: fraud funnels many cards into 
one device; legitimate sharing spreads one card across many devices.

## Suggestions

### Detection rules

A three-layer rule-based system was designed, directly derived from the 
analysis findings:

**Layer 1 - Transactional (<10ms):** blacklist checks (users, cards, devices; 
merchants excluded), chargeback history enforcement, amount and BIN-based scoring.

**Layer 2 - Entity (<30ms):** velocity checks (rapid-fire block, multi-card 
velocity within 60 min), multi-entity detection (user/device at multiple 
merchants, user with multiple cards/devices), rolling-window aggregates 
maintained in an in-memory state store. Some rules behave differently per 
period: multi-card velocity within 60 minutes is a hard block during 
Black Friday (63.9% fraud rate) but scored outside it (36.4%).

**Layer 3 - Relational (<50ms):** graph cluster membership lookup. 
Users connected to known fraud clusters through shared entities receive 
elevated risk scores. Compromised cards are flagged for monitoring even 
if their individual transactions look clean.

The system combines **hard-block rules** (for patterns with ≥7x fraud 
multiplier in both periods, where precision is inherently high) with 
**risk scoring** (weighted points per signal, cumulative threshold for 
approve/review/deny). This tiered approach reflects the error asymmetry: 
false negatives (missed fraud) are costlier than false positives (blocked 
legitimate transactions), so the system prioritizes recall while keeping 
precision above an acceptable floor through scoring. Thresholds adjust 
seasonally: tightened during Black Friday, relaxed during normal periods.

### Operational measures

**Blacklist management** with expiration policies and two tiers 
(hard block vs watch list), applying only to users, cards, and devices.

**Merchant remediation** instead of merchant blocking: merchants with 
sustained high chargeback rates enter remediation programs (mandatory 
3D Secure, lower transaction limits). Since merchants are CloudWalk's 
direct customers, the goal is to protect them from fraud exposure, not 
penalize them for being targeted.

**Rule performance monitoring** tracking true/false positive rates 
per rule, with champion-challenger testing for proposed changes.

**Decision audit trails** logging which rules fired, individual 
score contributions, and final outcomes, serving compliance, 
dispute handling, and rule tuning.

### Evolution path

The rule-based system addresses known fraud patterns with full 
transparency and auditability. The natural next step is a **hybrid 
system** where an ML scoring model runs in parallel, detecting 
subtle anomalies and emerging patterns that static rules miss. 
ML-detected patterns can then be codified as explicit rules, 
progressively strengthening the rule catalog while preserving 
explainability.

## Additional data that would improve detection

The analysis identified several gaps that limit detection capability 
with the current dataset alone:

**For direct detection improvement:** merchant category codes (MCC) 
to enable category-based velocity rules, IP/geolocation for 
distance-based checks and VPN detection, device fingerprint details 
for emulator/spoofing detection, and historical spending profiles 
for per-user deviation scoring.

**For investigation and response:** chargeback reason codes to 
separate true fraud from friendly fraud, 3D Secure authentication 
status, card lifecycle data (age, recent reports), and merchant 
risk profiles (platform tenure, chargeback history over time).

**For graph enrichment:** shared email/phone across accounts, 
shared billing/shipping addresses, and session-level data (IP, 
session tokens) - each adding new edges to the entity graph and 
increasing fraud ring detection coverage.

# Understand the industry

## Money flow, information flow and the role of the main players

### Money flow (clearing and settlement)

The money moves separately, after authorization:

1. At the end of the day (or batch cycle), the **acquirer** sends all authorized 
   transactions to the **card network** for clearing
2. The **network** calculates the net amounts owed between all acquirers and 
   issuers across all transactions
3. The **issuer** transfers the funds (minus interchange fees) to the **network**
4. The **network** distributes funds to the **acquirer** (minus network fees)
5. The **acquirer** deposits the funds into the **merchant's** account 
   (minus the acquirer's processing fee, known as the merchant discount rate)

Settlement typically takes 1-2 business days for debit and up to 30+ days for 
credit in Brazil, depending on the acquirer's contract with the merchant.

The cardholder pays their issuing bank later - either at the end of the billing 
cycle (credit) or immediately from their account (debit).

#### Fee distribution

At each step, a portion of the transaction amount is retained:

- **Interchange fee** - paid by the acquirer to the issuer (set by the card network)
- **Network fee** - retained by the card network for routing the transaction
- **Merchant discount rate (MDR)** - the total fee charged to the merchant by 
  the acquirer, which covers interchange, network fees, and the acquirer's margin

The merchant receives the transaction amount minus the MDR.

### Information flow (authorization)

When the cardholder initiates a purchase:

1. The **merchant** captures the card data and sends an authorization request 
   to the **acquirer**
2. The **acquirer** forwards the request to the **card network**
3. The **card network** routes it to the **issuer**
4. The **issuer** checks the cardholder's account (balance, credit limit, 
   fraud screening) and returns an approve or decline response
5. The response travels back: **issuer → network → acquirer → merchant**
6. The merchant completes or cancels the sale based on the response

This entire round trip typically happens in under 2 seconds.


### Main players

**Cardholder** - the person making the purchase. Holds a card issued by their bank.

**Merchant** - the business selling the product or service. Has a contract with an 
acquirer to accept card payments.

**Acquirer (acquiring bank)** - the financial institution on the merchant's side. 
It processes card transactions on behalf of the merchant, receives the authorization 
request, forwards it through the network, and ultimately deposits the funds into 
the merchant's account. Examples in Brazil: Cielo, Rede, Stone, Getnet.

**Card network (brand/scheme)** - Visa, Mastercard, Elo, etc. Sets the rules 
and standards for how transactions are processed, routes messages between acquirer 
and issuer, and arbitrates disputes. The network does not issue cards or hold funds - 
it operates the infrastructure that connects the two banks.

**Issuer (issuing bank)** - the bank on the cardholder's side. Issues the card, 
extends credit or manages the debit account, and decides whether to authorize 
a given transaction based on the cardholder's available balance, risk profile, 
and fraud controls.


## Main differences between acquirer, sub-acquirer and payment gateway

### Acquirer

An acquirer is a licensed financial institution that has a direct relationship with 
the card networks (Visa, Mastercard, etc.) and is authorized to process card 
transactions. The acquirer holds the merchant's funds, manages settlement, 
and bears financial responsibility in the payment chain - including liability 
for chargebacks if the merchant becomes insolvent.

The acquirer participates directly in the money flow: it receives funds from 
the issuer (via the network) and deposits them into the merchant's account.

To operate as an acquirer requires regulatory licensing, PCI DSS compliance, 
direct card network certification, and significant capital reserves. 
Examples in Brazil: Cielo, Rede, Stone, Getnet, and InfinitePay (CloudWalk).

### Sub-acquirer (payment facilitator)

A sub-acquirer sits between the merchant and the acquirer. It offers an all-in-one 
solution: the merchant signs a single contract with the sub-acquirer, which handles 
the technology, the relationship with acquirers and banks, risk management, 
and settlement.

The key difference: the sub-acquirer does not connect directly to card networks. 
It routes transactions through an acquirer behind the scenes. The merchant doesn't 
need to negotiate with acquirers, banks, or card networks - the sub-acquirer 
abstracts all of that complexity.

**How the flow changes:** an extra step is added. The transaction goes from 
merchant → sub-acquirer → acquirer → network → issuer (and back). The sub-acquirer 
participates in the money flow - it receives funds from the acquirer and then 
transfers them to the merchant.

**Tradeoffs:** simpler onboarding and single contract for the merchant, but 
typically higher fees (the sub-acquirer adds its own margin on top of the acquirer's). 
Some sub-acquirers redirect the customer to their own checkout page during payment, 
which can increase cart abandonment. Best suited for small and medium businesses 
that prioritize simplicity over cost optimization.

Examples in Brazil: PagSeguro, Mercado Pago.

### Payment gateway

A payment gateway is a **technology layer**, not a financial institution. It transmits 
transaction data between the merchant's checkout and the acquirer - it does not 
hold funds, process settlements, or participate in the money flow.

The gateway captures the card data securely, encrypts it, sends the authorization 
request to the acquirer, and returns the response to the merchant. That's it. 
The money still flows directly from acquirer to merchant.

**How the flow changes:** the gateway replaces the direct merchant → acquirer 
connection with a more flexible integration. The merchant can connect to 
multiple acquirers through a single gateway API, routing transactions to 
whichever acquirer offers better rates or availability (multi-acquirer strategy).

**Tradeoffs:** lower per-transaction costs than sub-acquirers and more control 
for the merchant, but the merchant must maintain separate contracts with 
acquirers, manage fraud prevention independently, and handle bank reconciliation. 
Best suited for larger businesses with the operational capacity to manage 
multiple provider relationships.

Examples in Brazil: Braspag (Cielo), MundiPagg.

## Chargebacks; how they differ from a cancellation and their connection with fraud

### What is a chargeback

A chargeback is a **forced reversal of a transaction**, initiated by the cardholder 
through their issuing bank. The cardholder contacts the issuer, claims the 
transaction was unauthorized, fraudulent, or otherwise invalid, and the issuer 
reverses the funds - pulling money back from the acquirer, who in turn debits 
the merchant's account.

The chargeback process follows card network rules (Visa, Mastercard) and involves 
a formal dispute with reason codes, evidence submission windows, and potential 
arbitration. The merchant can accept the chargeback or fight it by submitting 
evidence to the acquirer, who forwards it to the issuer for review. If the 
merchant's evidence is insufficient, the chargeback stands and the funds are 
permanently returned to the cardholder.

Beyond the lost sale, chargebacks carry additional costs for merchants: 
administrative fees, lost merchandise 
(rarely returned), and reputational damage with the acquirer. If a merchant's 
chargeback ratio exceeds card network thresholds, the acquirer faces fines 
and may terminate the merchant's account.

### What is a cancellation (refund/void)

A cancellation is a **voluntary reversal** initiated by the merchant or agreed 
upon between merchant and cardholder. It can take two forms:

- **Void** - if the transaction hasn't been settled yet, the merchant cancels it 
  before funds are transferred. No money moves.
- **Refund** - if the transaction has already settled, the merchant issues a 
  credit back to the cardholder's account through the normal payment flow.

In both cases, the process is cooperative - the merchant initiates it, the 
acquirer processes it, and there are no dispute codes, no fees, and no 
negative impact on the merchant's chargeback ratio.

### Connection to fraud

Chargebacks are directly connected to fraud in two ways:

**True fraud (third-party fraud)** - a criminal uses stolen card credentials 
to make unauthorized purchases. The legitimate cardholder discovers the charge 
and files a chargeback. This is the scenario the `has_cbk` flag in the dataset 
represents. In the acquiring world, these chargebacks are a direct cost of 
fraud that the acquirer and merchant absorb.

**Friendly fraud (first-party fraud)** - the legitimate cardholder makes a 
purchase, receives the product, but files a chargeback anyway - claiming the 
transaction was unauthorized or the product was never received. This is fraud 
by the cardholder against the merchant. It's harder to detect because the 
transaction itself looks legitimate.

For an acquirer, chargebacks are a critical metric. Card networks (Visa, 
Mastercard) monitor merchants' chargeback ratios and impose fines on 
acquirers that retain merchants with excessive rates. If the ratio stays 
above threshold, the acquirer may be forced to terminate the merchant's 
account. This creates a direct financial incentive for acquirers to invest 
in fraud prevention - reducing chargebacks protects both the merchant 
relationship and the acquirer's standing with the card networks.

## What is an Anti-Fraud System and How an Acquirer Uses It

### What it is

An anti-fraud system is a set of tools and processes designed to detect and 
prevent fraudulent transactions before they are approved - or, failing that, 
to identify them as quickly as possible afterward. The system sits in the 
authorization flow, between the merchant submitting a transaction and the 
acquirer forwarding it to the card network for approval.

At its core, an anti-fraud system evaluates each transaction and produces 
a decision: **approve**, **deny**, or **send to manual review**. It does this 
by analyzing transaction attributes (amount, time, location), entity behavior 
(user history, device patterns, card velocity), and contextual signals 
(merchant risk profile, seasonal patterns) against a set of rules, 
statistical models, or both.

### How an acquirer uses it

The acquirer deploys the anti-fraud system as a **pre-authorization filter**. 
Before forwarding a transaction to the card network for issuer approval, 
the acquirer's system evaluates it for fraud risk. Transactions flagged as 
high-risk are either denied outright (saving the acquirer from a future 
chargeback) or routed to manual review for analyst investigation.

In practice, an acquirer's anti-fraud system typically operates in layers:

**Rule-based layer** - predefined conditions that catch known fraud patterns. 
Fast, transparent, and easy to explain during audits. Examples: velocity limits, 
amount thresholds, blacklisted entities. This is the foundation described 
in this case study's solution design.

**Machine learning layer** - statistical models that score transactions based 
on patterns learned from historical data. Catches subtler anomalies that 
rules miss, but less transparent. Complements the rule layer rather than 
replacing it.

**Graph/relational layer** - maps connections between entities (users, cards, 
devices, merchants) to detect coordinated fraud and fraud rings that 
individual transaction analysis can't identify.

**Manual review** - human analysts investigate transactions that fall in 
the ambiguous zone between clear approve and clear deny. The anti-fraud 
system prioritizes the review queue by risk score.